### Librerias y Paquetes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pymcel as pym 
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

### Efemerides de las lunas de Jupiter (Io, Europa, Ganimedes, Calisto)

Se consultan efemerides con `pym.consulta_horizons` y se integran con `pym.ncuerpos_solucion` siguiendo el ejemplo del otro notebook.

In [ ]:
# Ephemerides from JPL Horizons (positions/velocities in SI)
_, _, X_io = pym.consulta_horizons(id='501', location='@5', epochs='2024-01-01 00:00:00') # ID 501 corresponde a Io, pero se puede usar 'Io' también
_, _, X_eu = pym.consulta_horizons(id='502', location='@5', epochs='2024-01-01 00:00:00') # ID 502 corresponde a Europa
_, _, X_ga = pym.consulta_horizons(id='503', location='@5', epochs='2024-01-01 00:00:00') # ID 503 corresponde a Ganymede
_, _, X_ca = pym.consulta_horizons(id='504', location='@5', epochs='2024-01-01 00:00:00') # ID 504 corresponde a Callisto
_, _, X_J = pym.consulta_horizons(id='599', location='@5', epochs='2024-01-01 00:00:00') # ID 599 corresponde a Jupiter
_, _, X_sol = pym.consulta_horizons(id='Sun', location='@5', epochs='2024-01-01 00:00:00') # ID Sun corresponde al Sol

state_vectors = {
    'Io': X_io,
    'Europa': X_eu,
    'Ganymede': X_ga,
    'Callisto': X_ca,
    'Jupiter': X_J,
    'Sun': X_sol,
}

rows = []
for body, vec in state_vectors.items():
    arr = np.asarray(vec)
    if arr.ndim == 1 and arr.size == 6:
        arr = arr.reshape(1, 6)
    if arr.ndim != 2 or arr.shape[1] != 6:
        raise ValueError(f"Vector de estado inesperado para {body}: shape {arr.shape}")
    for i, (x, y, z, vx, vy, vz) in enumerate(arr):
        rows.append({
            'body': body,
            'row': i,
            'x': x,
            'y': y,
            'z': z,
            'vx': vx,
            'vy': vy,
            'vz': vz,
        })

df_state = pd.DataFrame(rows)
df_state

In [ ]:
import requests
import re

bodies_ids = {
    'Io': '501',
    'Europa': '502',
    'Ganymede': '503',
    'Callisto': '504',
    'Jupiter': '599',
    'Sun': 'Sun',
}

def get_mass_si(command_id):
    """Consulta la masa o GM desde JPL Horizons y devuelve la masa en kg."""
    url = 'https://ssd.jpl.nasa.gov/api/horizons.api'
    params = {'format':'json', 'COMMAND':command_id, 'OBJ_DATA':'YES', 'MAKE_EPHEM':'NO'}
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        text = r.json().get('result', '')
    except Exception:
        return None

    m1 = re.search(r'Mass\s+x\s+10\^(\d+)\s+\(kg\)\s*=\s*([~]?[\d\.]+)', text)
    if m1:
        return float(m1.group(2).replace('~','')) * (10**float(m1.group(1)))

    m2 = re.search(r'Mass\s+\(10\^(\d+)\s+kg\)\s*=\s*([~]?[\d\.]+)', text)
    if m2:
        return float(m2.group(2).replace('~','')) * (10**float(m2.group(1)))

    m3 = re.search(r'Mass,\s+10\^(\d+)\s+kg\s*=\s*([~]?[\d\.]+)', text)
    if m3:
        return float(m3.group(2).replace('~','')) * (10**float(m3.group(1)))

    mg = re.search(r'GM\s+\(km\^3/s\^2\)\s*=\s*([~]?[\d\.]+)', text)
    if mg:
        G_approx = 6.67430e-11
        gm_km3s2 = float(mg.group(1).replace('~',''))
        gm_m3s2 = gm_km3s2 * 1e9
        return gm_m3s2 / G_approx

    return None

data_mass = []
for name, horizon_id in bodies_ids.items():
    mass = get_mass_si(horizon_id)
    data_mass.append({'Cuerpo': name, 'ID Horizons': horizon_id, 'Masa (kg)': mass})

df_mass = pd.DataFrame(data_mass)
pd.options.display.float_format = '{:.4e}'.format
df_mass

In [ ]:
df_final = pd.merge(df_state, df_mass, left_on='body', right_on='Cuerpo', how='left')
df_final.drop(columns=['Cuerpo', 'ID Horizons'], inplace=True)

G = 6.67430e-11
M_J = 1.989e30 # Masa del Sol, que es la dominante en el sistema
U_L = np.linalg.norm(X_sol[:3] - X_J[:3], axis=0)
U_T = 2 * 60       # 2 minutos es un tiempo típico de interacción en mareas extremas
U_M = U_L**3 / (G * U_T**2)
print(f"Unidad de tiempo (U_T): {U_T} s")
print(f"Unidad de longitud (U_L): {U_L} m")
print(f"Unidad de masa (U_M): {U_M} kg")

df_canonical = df_final.copy()
df_canonical['x'] /= U_L
df_canonical['y'] /= U_L
df_canonical['z'] /= U_L
df_canonical['vx'] /= (U_L / U_T)
df_canonical['vy'] /= (U_L / U_T)
df_canonical['vz'] /= (U_L / U_T)
df_canonical['Masa (kg)'] /= U_M

sistema = [dict(m=df_canonical['Masa (kg)'][i],
                r=np.array([df_canonical['x'][i], df_canonical['y'][i], df_canonical['z'][i]]),
                v=np.array([df_canonical['vx'][i], df_canonical['vy'][i], df_canonical['vz'][i]]))
           for i in range(len(df_canonical))]

T = 10 * 42.5 * 3600/U_T # 42.5 horas es el período orbital de Io, que es el más rápido del sistema
N = 12750
ts = np.linspace(0, T, N)

rs, vs, rcms, vcms, cons = pym.ncuerpos_solucion(sistema, ts)
rs.shape


In [ ]:
cons['E']

In [ ]:
df_canonical

In [ ]:
# Gráfico 3D de las órbitas con respecto al centro de masa
fig = go.Figure()

# Nombres en el mismo orden de sistema
bodies = df_final['body'].tolist()

# Añadir las trayectorias de cada cuerpo
for i in range(len(sistema)-1):
    nombre = bodies[i] if i < len(bodies) else f'Cuerpo {i+1}'
    # Trayectoria
    fig.add_trace(go.Scatter3d(
        x=rcms[i, :, 0],
        y=rcms[i, :, 1],
        z=rcms[i, :, 2],
        mode='lines',
        name=nombre,
        line=dict(width=3)
    ))

    # Punto inicial
    fig.add_trace(go.Scatter3d(
        x=[rcms[i, 0, 0]],
        y=[rcms[i, 0, 1]],
        z=[rcms[i, 0, 2]],
        mode='markers',
        name=f'Inicio {nombre}',
        marker=dict(size=6, symbol='circle'),
        showlegend=False
    ))

    # Punto final
    fig.add_trace(go.Scatter3d(
        x=[rcms[i, -1, 0]],
        y=[rcms[i, -1, 1]],
        z=[rcms[i, -1, 2]],
        mode='markers+text',
        name=f'Fin {nombre}',
        text=[nombre],
        textposition='top center',
        marker=dict(size=6, symbol='x'),
        showlegend=False
    ))

# Centro de masa (origen)
fig.add_trace(go.Scatter3d(
    x=[0],
    y=[0],
    z=[0],
    mode='markers',
    name='Centro de Masa',
    marker=dict(size=10, color='black', symbol='diamond')
))

fig.update_layout(
    title='Órbitas con respecto al Centro de Masa',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='cube'
    ),
    width=900,
    height=700
 )

fig.show()

In [ ]:
rcms_J = np.array([rcms[i] - rcms[4] for i in range(5)])



In [ ]:
# Gráfico 3D de las órbitas con respecto al centro de masa
fig = go.Figure()

# Nombres en el mismo orden de sistema
bodies = df_final['body'].tolist()

# Añadir las trayectorias de cada cuerpo
for i in range(len(sistema)-1):
    nombre = bodies[i] if i < len(bodies) else f'Cuerpo {i+1}'
    # Trayectoria
    fig.add_trace(go.Scatter3d(
        x=rcms_J[i, :, 0],
        y=rcms_J[i, :, 1],
        z=rcms_J[i, :, 2],
        mode='lines',
        name=nombre,
        line=dict(width=3)
    ))

    # Punto inicial
    fig.add_trace(go.Scatter3d(
        x=[rcms_J[i, 0, 0]],
        y=[rcms_J[i, 0, 1]],
        z=[rcms_J[i, 0, 2]],
        mode='markers',
        name=f'Inicio {nombre}',
        marker=dict(size=6, symbol='circle'),
        showlegend=False
    ))

    # Punto final
    fig.add_trace(go.Scatter3d(
        x=[rcms_J[i, -1, 0]],
        y=[rcms_J[i, -1, 1]],
        z=[rcms_J[i, -1, 2]],
        mode='markers+text',
        name=f'Fin {nombre}',
        text=[nombre],
        textposition='top center',
        marker=dict(size=6, symbol='x'),
        showlegend=False
    ))

# Centro de masa (origen)
fig.add_trace(go.Scatter3d(
    x=[0],
    y=[0],
    z=[0],
    mode='markers',
    name='Centro de Masa',
    marker=dict(size=10, color='black', symbol='diamond')
))

fig.update_layout(
    title='Órbitas con respecto al Centro de Masa',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='cube'
    ),
    width=900,
    height=700
 )

fig.show()

In [ ]:
# Animación 3D de todos los cuerpos
T = 20

# Reducir cantidad de frames para que sea fluida
step_frame = 100
frame_indices = np.arange(0, len(ts), step_frame)
num_frames = len(frame_indices)
t_frames = ts[frame_indices]

# Colores para los cuerpos
colores = ['blue', 'green', 'red', 'orange', 'purple', 'cyan', 'magenta']

# Crear frames: cada frame contiene TODOS los cuerpos
frames_data_full = []
for frame_id, j in enumerate(frame_indices):
    frame_traces = []

    for i in range(len(rcms_J)):
        # Trayectoria acumulada del cuerpo i
        frame_traces.append(go.Scatter3d(
            x=rcms_J[i, :j+1, 0].tolist(),
            y=rcms_J[i, :j+1, 1].tolist(),
            z=rcms_J[i, :j+1, 2].tolist(),
            mode='lines',
            line=dict(color=colores[i % len(colores)], width=3),
            name=f'Cuerpo {i+1} (m={sistema[i]["m"]})',
            showlegend=(frame_id == 0)
        ))

        # Posición actual del cuerpo i
        frame_traces.append(go.Scatter3d(
            x=[float(rcms_J[i, j, 0])],
            y=[float(rcms_J[i, j, 1])],
            z=[float(rcms_J[i, j, 2])],
            mode='markers',
            marker=dict(size=6, color=colores[i % len(colores)]),
            name=f'Posición {i+1}',
            showlegend=False
        ))

    # Centro de masa
    frame_traces.append(go.Scatter3d(
        x=[0.0],
        y=[0.0],
        z=[0.0],
        mode='markers',
        marker=dict(size=10, color='black', symbol='diamond'),
        name='Centro de Masa',
        showlegend=(frame_id == 0)
    ))

    frames_data_full.append(
        go.Frame(
            data=frame_traces,
            name=str(frame_id),
            traces=list(range(2 * len(sistema) + 1))
        )
    )

# Slider
s_steps = []
for frame_id in range(num_frames):
    s_steps.append(dict(
        method='animate',
        args=[[str(frame_id)], dict(
            mode='immediate',
            frame=dict(duration=100, redraw=True),
            transition=dict(duration=50)
        )],
        label=f'{t_frames[frame_id]:.2f}s'
    ))

sliders = [dict(
    steps=s_steps,
    active=0,
    currentvalue={'prefix': 'Tiempo: '},
    pad={'t': 50}
)]

# Botones
updatemenus = [
    dict(
        type='buttons',
        buttons=[
            dict(label='▶ Play', method='animate',
                 args=[None, {'frame': {'duration': 100, 'redraw': True},
                              'fromcurrent': True,
                              'transition': {'duration': 50}}]),
            dict(label='⏸ Pausa', method='animate',
                 args=[[None], {'frame': {'duration': 0, 'redraw': True},
                                'mode': 'immediate',
                                'transition': {'duration': 0}}])
        ],
        direction='left',
        pad={'r': 10, 't': 87},
        showactive=False,
        x=0.1,
        xanchor='right',
        y=0.2,
        yanchor='top'
    ),

    dict(
        type="buttons",
        buttons=[
            dict(label="Lento",
                 method="animate",
                 args=[None, {"frame": {"duration": 250, "redraw": True}, "fromcurrent": True, "transition": {"duration": 150, "easing": "quadratic-in-out"}}]),
            dict(label="Normal",
                 method="animate",
                 args=[None, {"frame": {"duration": 100, "redraw": True}, "fromcurrent": True, "transition": {"duration": 50, "easing": "quadratic-in-out"}}]),
            dict(label="Rápido",
                 method="animate",
                 args=[None, {"frame": {"duration": 20, "redraw": True}, "fromcurrent": True, "transition": {"duration": 20, "easing": "quadratic-in-out"}}])
        ],
        direction="right",
        pad={"r": 10, "t": 87},
        showactive=True,
        x=0.9,
        xanchor="right",
        y=0.2,
        yanchor="top"
    )
]

# Trazas iniciales: deben coincidir con las trazas de cada frame
initial_data = []
for i in range(len(sistema)):
    initial_data.append(go.Scatter3d(
        x=[], y=[], z=[],
        mode='lines',
        line=dict(color=colores[i % len(colores)], width=3),
        name=f'Cuerpo {i+1} (m={sistema[i]["m"]})'
    ))
    initial_data.append(go.Scatter3d(
        x=[], y=[], z=[],
        mode='markers',
        marker=dict(size=6, color=colores[i % len(colores)]),
        name=f'Posición {i+1}',
        showlegend=False
    ))

initial_data.append(go.Scatter3d(
    x=[0.0], y=[0.0], z=[0.0],
    mode='markers',
    marker=dict(size=10, color='black', symbol='diamond'),
    name='Centro de Masa'
))

fig = go.Figure(
    data=initial_data,
    layout=go.Layout(
        title=f'Animación 3D de {len(sistema)} cuerpos',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data'
        ),
        updatemenus=updatemenus,
        sliders=sliders,
        width=900,
        height=700,
        scene_camera_eye=dict(x=1.8, y=1.8, z=0.8)
    ),
    frames=frames_data_full
)

fig.show()

## Consulta de las efemerides de IO con `consulta_horizons`.

**Se calcula el tiempo necesario para que pasen 10 orbitas de IO**

In [ ]:
from astropy.time import Time
import astropy.units as u

t0 = Time("2024-01-01 00:00:00", scale="utc")
t1 = t0 + 10*42.5*u.hour

print(t1.isot)

In [ ]:
int(10*42.5/24), int(0.708333333333332*24)

In [ ]:
dicfecha = dict(start='2024-01-01 00:00:00', stop='2024-01-18 17:00:00', step='2m')

In [ ]:
# Sellaman las efeemrides de JPL Horizons para el período de interés, con un paso de 2 minutos

_,_, X_io = pym.consulta_horizons(id='501', location='@5', epochs=dicfecha) # ID 501 corresponde a Io, pero se puede usar 'Io' también

In [ ]:
X_io.shape

In [ ]:
rs.shape

In [ ]:
rs_JPL_IO = X_io.T[:3]
rs_JPL_IO_1 = rs_JPL_IO.T[1:]
rs_JPL_IO_2 = rs_JPL_IO.T[0:-1]

In [ ]:
rs_JPL_IO_1.shape, rs_JPL_IO_2.shape

In [ ]:
rs_JPL_IO[:10]/U_L

In [ ]:
rcms_IO = rcms_J[0]

In [ ]:
rcms_IO.shape, rs_JPL_IO_1.shape, ts.shape

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(ts, np.linalg.norm(rs_JPL_IO_1/1e3 - rcms_IO*U_L/1e3, axis=1), label='diferencia')
plt.xlabel('Tiempo (U_T)')
plt.ylabel('Distancia (km)')
plt.title('Diferencia entre posición simulada con N-Cuerpos y JPL Horizons para Io')
plt.legend()

In [ ]:
cons['U']

In [ ]:
import numpy as np

# Toma U desde Cons o cons (por si hay diferencia de mayúsculas)
if 'Cons' in globals() and isinstance(cons, dict):
    U = np.asarray(cons['U'])
elif 'cons' in globals() and isinstance(cons, dict):
    U = np.asarray(cons['U'])
else:
    raise NameError("No existe 'Cons' ni 'cons' con la clave 'U'.")

# Toma K desde cons (o Cons como respaldo)
if 'cons' in globals() and isinstance(cons, dict):
    K = np.asarray(cons['K'])
elif 'Cons' in globals() and isinstance(cons, dict):
    K = np.asarray(cons['K'])
else:
    raise NameError("No existe 'cons' ni 'Cons' con la clave 'K'.")

# Energía total instantánea
E = K + U

# Conservación de energía: drift relativo entre inicio y final
if E.ndim == 0:
    drift_rel = 0.0
else:
    E0 = float(E[0])
    Ef = float(E[-1])
    drift_rel = abs(Ef - E0) / max(abs(E0), 1e-30)

# Virial para sistema gravitacional ligado: 2<K> + <U> ~ 0
K_med = float(np.mean(K))
U_med = float(np.mean(U))
virial_resid = 2.0 * K_med + U_med
virial_rel = abs(virial_resid) / max(abs(U_med), 1e-30)

# Criterios prácticos
energia_conservada = drift_rel < 1e-3   # 0.1%
virial_cumple = virial_rel < 0.1        # 10%
sistema_ligado = (K_med + U_med) < 0 and virial_cumple

print(f"<K> = {K_med:.6e}")
print(f"<U> = {U_med:.6e}")
print(f"<E> = {K_med + U_med:.6e}")
print(f"Drift relativo de E = {drift_rel:.3e}")
print(f"Residual virial relativo = {virial_rel:.3e}")
print(f"Energía conservada?: {'Sí' if energia_conservada else 'No'}")
print(f"Sistema ligado (E<0 y virial)?: {'Sí' if sistema_ligado else 'No'}")